# Lazy image read example

Start `examples/python/readWrite/testDevice.py` in another process, then run this notebook. It connects to the readWrite `TestDevice`, reads the two image channels, and explicitly pulls the lazy BinaryData-backed image payload into the notebook.

In [ ]:
from pathlib import Path

import stipy


config = stipy.Configuration({
    "Device Name": "TestDevice Notebook Client",
    "IP Address": "localhost",
    "Module": "0",
    "Target Server": "sr-magis/2/Frame2",
})

config.set("NetworkHub", "NameService", "192.168.88.252:2809")
config.set("omniORB", "traceLevel", "0")
config.set("omniORB", "scanGranularity", "1")
config.set("omniORB", "clientConnectTimeOutPeriod", "500")
config.set("omniORB", "clientCallTimeOutPeriod", "2000")

device_id = stipy.DeviceID("TestDevice", "localhost", 0, "sr-magis/2/Frame2")
device = stipy.connect(device_id, config=config)
device

Channels 13 and 14 both return `Image`. Channel 13 is backed by inline `BinaryData`; channel 14 is backed by a `FileHolder`. Remote reads return lightweight objects first, so the BinaryData-backed image can report its size before the bytes are pulled.

In [ ]:
binary_image_read = device.read(13)
file_image_read = device.read(14)

binary_image_read, file_image_read

The direct read result is lazy. The remote channel cache also stores the read result as `lastMeasurement`, using the same stream-backed representation for heavy BinaryData/Image channel state.

In [ ]:
def last_image(channel_number):
    channel = device.getChannelManager().getChannel(channel_number)
    measurement = channel.getLastMeasurement()
    assert measurement.getType() == stipy.MixedValueType.Image
    return measurement.getImage()


direct_binary_image = binary_image_read
direct_file_reference_image = file_image_read
cached_binary_image = last_image(13)
cached_file_reference_image = last_image(14)

direct_binary_image, direct_file_reference_image, cached_binary_image, cached_file_reference_image

In [ ]:
def image_summary(image):
    data = image.getData() if image.hasData() else None
    file_id = image.getFileID()
    summary = {
        "width": image.getWidth(),
        "height": image.getHeight(),
        "hasData": image.hasData(),
        "hasFile": image.hasFile(),
        "fileID": {
            "origin": file_id.origin,
            "path": file_id.path,
            "filename": file_id.filename,
            "persistenceLocation": file_id.persistenceLocation,
        },
    }
    if data is not None:
        summary["data"] = {
            "bytes": data.bytes(),
            "length": data.length(),
            "wordsize": data.wordsize(),
            "hasStream": data.hasStream(),
            "hasLocalData": data.hasLocalData(),
            "isMaterialized": data.isMaterialized(),
        }
    return summary


{
    "directBinaryImage": image_summary(direct_binary_image),
    "directFileImage": image_summary(direct_file_reference_image),
    "cachedBinaryImage": image_summary(cached_binary_image),
    "cachedFileImage": image_summary(cached_file_reference_image),
}

Explicitly pull the direct BinaryData-backed image read. Before `pull()`, the object reports stream metadata without local data. After `pull()`, the bytes are available in the notebook.

In [ ]:
binary_data = direct_binary_image.getData()

before = {
    "bytes": binary_data.bytes(),
    "wordsize": binary_data.wordsize(),
    "hasStream": binary_data.hasStream(),
    "hasLocalData": binary_data.hasLocalData(),
    "isMaterialized": binary_data.isMaterialized(),
}

pulled = binary_data.pull()
payload = binary_data.getBytes()

after = {
    "pullSucceeded": pulled,
    "payloadLength": len(payload),
    "hasLocalData": binary_data.hasLocalData(),
    "isMaterialized": binary_data.isMaterialized(),
    "first16Bytes": list(payload[:16]),
}

before, after

In [ ]:
output_path = Path("pulled-readWrite-channel-13.raw")
output_path.write_bytes(payload)
output_path.resolve()

Optional display cell. The example image is raw 10x10 grayscale data, so this reshapes the pulled bytes into a small pixel array if NumPy and Matplotlib are available.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    pixels = np.frombuffer(payload, dtype=np.uint8).reshape(
        direct_binary_image.getHeight(),
        direct_binary_image.getWidth(),
    )
    plt.imshow(pixels, cmap="gray", vmin=0, vmax=255)
    plt.axis("off")
except ImportError:
    print("Install numpy and matplotlib to display the image inline.")

The FileHolder-backed image carries file identity metadata rather than inline lazy BinaryData.

In [ ]:
image_summary(direct_file_reference_image)